<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="http://www.uoc.edu/portal/_resources/common/imatges/marca_UOC/UOC_Masterbrand.jpg", align="left">
</div>
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.877 · Anàlisi de sentiments i textos</p>
<p style="margin: 0; text-align:right;">Màster universitari de Ciències de Dades (Data science)</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Estudis d'Informàtica, Multimèdia i Telecomunicacions</p>
</div>
</div>
<div style="width: 100%; clear: both;">
<div style="width:100%;">&nbsp;</div>

# Mòdul 5: Deep learning per a l'anàlisi de sentiments

## Anàlisi de sentiments per aspectes

En aquest notebook, entrenarem un mòdul per detectar aspectes en opinions (aspect term extraction) i, un altre, per analitzar sentiments (polaritats) en aspectes d'opinions (targeted-aspect based sentiment analysis).


En tots dos casos, treballarem amb dades de la tasca d'AbSA de SemEval 2014 http://alt.qcri.org/semeval2014/task4/.



En primer lloc, carreguem les llibreries necessàries.

In [1]:
from torchtext import data
import torch


import torch.optim as optim
import torch.nn as nn
import time

# Detecció d'aspectes en opinions

Per poder carregar les dades en un dataset de `torchtext`, primer, hem de definir els camps. En aquest cas, necessitem dos camps, un per al text i un altre per a l'aspecte.

In [2]:
TEXT = data.Field(tokenize = 'spacy', include_lengths=True)
ASPECT = data.LabelField()

/home/emilia/Projects/pytorch-sentiment-analysis/venv/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/emilia/Projects/pytorch-sentiment-analysis/venv/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/emilia/Projects/pytorch-sentiment-analysis/venv/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.i

Les dades ja estan dividides en tres datasets ('restaurants_aspect_train.csv', 'restaurants_aspect_val' i 'restaurants_aspect_test.csv'). L'objectiu és identificar cada comentari amb un dels cinc aspectes següents:

- food / ambience / service / price / anecdotes miscellaneous

Si carreguem un dataset, veiem en què consisteix:

In [3]:
import pandas as pd

df = pd.read_csv('restaurants_category_train.csv')

df.head()

,sid,text,aid,category,polarity
0,573,the wine the service was very good too,573-2,service,1
1,3364,this is the perfect date spot for williamsburg...,3364-1,anecdotes miscellaneous,1
2,766,the dim sum however was very good,766-1,food,1
3,1913,worth it for a special occasion or any time,1913-1,anecdotes miscellaneous,1
4,780,overall worht every penny,780-1,price,1


Aleshores, definim la variable `fields` següent per poder carregar les dades.

In [4]:
fields = [(None, None), ('text', TEXT), (None, None), ('aspect', ASPECT)]


I les podem carregar directament de la manera següent.

In [5]:
train_data, valid_data, test_data = data.TabularDataset.splits(
                                        path = '.',
                                        train = 'restaurants_category_train.csv',
                                        validation = 'restaurants_category_val.csv',
                                        test = 'restaurants_category_test.csv',
                                        format = 'csv',
                                        fields = fields,
                                        skip_header = True)

Podem visualitzar algun exemple per confirmar la càrrega correcta.

In [6]:
print(vars(train_data.examples[10]))
print(vars(valid_data.examples[9]))
print(vars(test_data.examples[0]))

{'text': ['indoor', 'was', 'very', 'cozy', 'and', 'cute'], 'aspect': 'ambience'}
{'text': ['the', 'service', 'was', 'bad', 'the', 'food', 'took', 'to', 'forever', 'to', 'come', 'we', 'sat', 'on', 'the', 'upper', 'level'], 'aspect': 'service'}
{'text': ['the', 'bread', 'is', 'top', 'notch', 'as', 'well'], 'aspect': 'food'}


Construïm els vocabularis de la mateixa manera que hem vist en el mòdul d'Anàlisi de sentiments.

In [7]:
MAX_VOCAB_SIZE = 25_000

TEXT.build_vocab(train_data, 
                 vectors = "glove.6B.300d",
                 unk_init = torch.Tensor.normal_)

ASPECT.build_vocab(train_data)

I explorem el resultat.

In [8]:
print(len(TEXT.vocab))

print(len(ASPECT.vocab))
print(ASPECT.vocab.freqs)

3587
5
Counter({'food': 867, 'anecdotes miscellaneous': 578, 'service': 438, 'ambience': 299, 'price': 232})


De nou, com en el notebook d'Anàlisi de sentiments, definim un `BucketIterator` per a cada dataset de la manera següent:

In [9]:
BATCH_SIZE = 16

# si tenim GPU disponible, s'hi computaran les dades
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

train_iterator, valid_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, valid_data, test_data), 
    batch_size = BATCH_SIZE,
    sort_key=lambda x: len(x.text),
    sort_within_batch=False,
    device = device)

## Definim el model

In [10]:
import torch.nn as nn

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, 
                 bidirectional, dropout, pad_idx):
        
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx = pad_idx)
        
        self.rnn = nn.LSTM(embedding_dim, 
                           hidden_dim, 
                           num_layers=n_layers, 
                           bidirectional=bidirectional, 
                           dropout=dropout)
        
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text, text_lengths):
              
        with torch.no_grad():
            embedded = self.dropout(self.embedding(text))
                
        #pack sequence
        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, text_lengths, enforce_sorted=False)
        
        packed_output, (hidden, cell) = self.rnn(packed_embedded)
        
        output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output)
        
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim = 1))
                            
        return self.fc(hidden)

In [11]:
INPUT_DIM = len(TEXT.vocab)
EMBEDDING_DIM = 300
HIDDEN_DIM = 256
OUTPUT_DIM = 5
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
PAD_IDX = TEXT.vocab.stoi[TEXT.pad_token]

model = BiLSTM(INPUT_DIM, 
            EMBEDDING_DIM, 
            HIDDEN_DIM, 
            OUTPUT_DIM, 
            N_LAYERS, 
            BIDIRECTIONAL, 
            DROPOUT, 
            PAD_IDX)


## Preparem l'entrenament

In [12]:
import torch.optim as optim


optimizer = optim.SGD(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()
model = model.to(device)
criterion = criterion.to(device)


Definim una funció auxiliar per calcular l'accuracy.

In [13]:
def categorical_accuracy(preds, y):
    max_preds = preds.argmax(dim = 1, keepdim = True) # get the index of the max probability
    correct = max_preds.squeeze(1).eq(y) # check if it is correct
    return correct.sum() / torch.FloatTensor([y.shape[0]])

I les funcions per als loops d'entrenament i d'avaluació.

In [14]:
def train(model, iterator, optimizer, criterion):
    
    epoch_loss = 0
    epoch_acc = 0
    model.train()

    for batch in iterator:
        optimizer.zero_grad()
        text, text_lengths = batch.text
        predictions = model(text, text_lengths).squeeze(1)   
        loss = criterion(predictions, batch.aspect)
        acc = categorical_accuracy(predictions, batch.aspect)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion):
    
    epoch_loss = 0
    epoch_acc = 0    
    model.eval()
    
    with torch.no_grad():    
        for batch in iterator:
            text, text_lengths = batch.text
            predictions = model(text, text_lengths).squeeze(1)  
            loss = criterion(predictions, batch.aspect)
            acc = categorical_accuracy(predictions, batch.aspect)
            epoch_loss += loss.item()
            epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)


import time

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

# Entrenem el model

In [15]:
N_EPOCHS = 10


best_valid_loss = float('inf')


for epoch in range(N_EPOCHS):

    start_time = time.time()
    
    train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, valid_iterator, criterion)
    
    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'absa1.pt')
    
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

Epoch: 01 | Epoch Time: 0m 1s
	Train Loss: 1.574 | Train Acc: 32.96%
	 Val. Loss: 1.535 |  Val. Acc: 34.82%
Epoch: 02 | Epoch Time: 0m 1s
	Train Loss: 1.525 | Train Acc: 35.86%
	 Val. Loss: 1.501 |  Val. Acc: 34.82%
Epoch: 03 | Epoch Time: 0m 1s
	Train Loss: 1.504 | Train Acc: 35.90%
	 Val. Loss: 1.485 |  Val. Acc: 34.82%
Epoch: 04 | Epoch Time: 0m 1s
	Train Loss: 1.493 | Train Acc: 35.91%
	 Val. Loss: 1.474 |  Val. Acc: 34.82%
Epoch: 05 | Epoch Time: 0m 1s
	Train Loss: 1.487 | Train Acc: 35.92%
	 Val. Loss: 1.465 |  Val. Acc: 34.82%
Epoch: 06 | Epoch Time: 0m 1s
	Train Loss: 1.477 | Train Acc: 35.99%
	 Val. Loss: 1.456 |  Val. Acc: 34.82%
Epoch: 07 | Epoch Time: 0m 2s
	Train Loss: 1.469 | Train Acc: 36.09%
	 Val. Loss: 1.446 |  Val. Acc: 35.15%
Epoch: 08 | Epoch Time: 0m 1s
	Train Loss: 1.460 | Train Acc: 37.23%
	 Val. Loss: 1.436 |  Val. Acc: 38.94%
Epoch: 09 | Epoch Time: 0m 1s
	Train Loss: 1.456 | Train Acc: 38.07%
	 Val. Loss: 1.426 |  Val. Acc: 40.09%
Epoch: 10 | Epoch Time: 0m 1

# Avaluem el model

In [16]:
model = BiLSTM(INPUT_DIM, 
            EMBEDDING_DIM, 
            HIDDEN_DIM, 
            OUTPUT_DIM, 
            N_LAYERS, 
            BIDIRECTIONAL, 
            DROPOUT, 
            PAD_IDX)

pretrained_embeddings = TEXT.vocab.vectors
model.embedding.weight.data.copy_(pretrained_embeddings)

UNK_IDX = TEXT.vocab.stoi[TEXT.unk_token]

model.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
model.embedding.weight.data[PAD_IDX] = torch.zeros(EMBEDDING_DIM)

model.to(device)

BiLSTM(
  (embedding): Embedding(3587, 300, padding_idx=1)
  (rnn): LSTM(300, 256, num_layers=2, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=512, out_features=5, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [17]:
model.load_state_dict(torch.load('absa1.pt'))

test_loss, test_acc = evaluate(model, test_iterator, criterion)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')


Test Loss: 1.405 | Test Acc: 44.48%


# Anàlisi de sentiments per aspectes (targeted)

In [18]:
import numpy as np
from transformers import BertModel
from transformers import BertTokenizer

bert = BertModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [19]:
TEXT = data.Field(sequential = False, use_vocab=False)
TERM = data.Field(sequential = False, use_vocab=False)
POLARITY = data.LabelField()


fields = [(None, None), ('text', TEXT), (None, None), ('term', TERM), ('polarity', POLARITY)] 

In [20]:
train_data, valid_data, test_data = data.TabularDataset.splits(
                                        path = '.',
                                        train = 'restaurants_term_train.csv',
                                        validation = 'restaurants_term_val.csv',
                                        test = 'restaurants_term_test.csv',
                                        format = 'csv',
                                        fields = fields,
                                        skip_header = True)

In [21]:
def tokenitzar_i_indexar_amb_bert(dataset, tokenizer):
    maxlen = 20
    for i in range(len(dataset.examples)):
        
        text = dataset.examples[i].text
        term = dataset.examples[i].term
        text_raw_sequence = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text))
        term_raw_sequence = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(term))
        

        if len(term_raw_sequence)>=int(maxlen/4):
            term_raw_sequence = term_raw_sequence[0:int(maxlen/4)]
        seq = text_raw_sequence[:maxlen-3-len(term_raw_sequence)]

        text_bert_ids = tokenizer.convert_tokens_to_ids(["[CLS]"])\
            + seq +  tokenizer.convert_tokens_to_ids(["[SEP]"])\
            + term_raw_sequence + tokenizer.convert_tokens_to_ids(["[SEP]"])

        bert_segments_ids = [0] * (len(seq)+2) + [1] * (len(term_raw_sequence) + 1) + [0]*(maxlen-len(text_bert_ids))

        text_bert_ids = text_bert_ids + [0]*(maxlen-len(text_bert_ids))
        new_text = [text_bert_ids, bert_segments_ids]
        
        dataset.examples[i].text = np.asarray(text_bert_ids, dtype='int64')
        dataset.examples[i].term = np.asarray(bert_segments_ids, dtype='int64')



tokenitzar_i_indexar_amb_bert(train_data, tokenizer)
tokenitzar_i_indexar_amb_bert(valid_data, tokenizer)
tokenitzar_i_indexar_amb_bert(test_data, tokenizer)

In [22]:
#TEXT.build_vocab()
POLARITY.build_vocab(train_data)

POLARITY.vocab.freqs

Counter({'-1': 637, '1': 1740})

In [23]:
BATCH_SIZE = 16

# si tenim GPU disponible, s'hi computaran les dades
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

train_iterator, valid_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, valid_data, test_data), 
    batch_size = BATCH_SIZE,
    sort_key=lambda x: len(x.text),
    sort_within_batch=False,
    device = device)

In [24]:
args = {'bert_dim' : 768, 'dropout': 0.1} #768 és la hidden size d'aquest model BERT
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class BERT_classifier(nn.Module):
    def __init__(self, bert, args):
        super(BERT_classifier, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(args['dropout'])
        self.dense = nn.Linear(args['bert_dim'], 2)

    def forward(self, inputs):
        text_bert_indices, bert_segments_ids = inputs.text, inputs.term
        _, pooled_output = self.bert(text_bert_indices, bert_segments_ids)
        pooled_output = self.dropout(pooled_output)
        logits = self.dense(pooled_output)
        return logits

model = BERT_classifier(bert, args).to(device)

Definim les funcions `train` i `evaluate`.

In [25]:
def train(model, iterator, optimizer, criterion):
    
    epoch_loss = 0
    epoch_acc = 0
    
    model.train() 
    
    for batch in iterator:
        optimizer.zero_grad()  
        predictions = model(batch)   
        loss = criterion(predictions, batch.polarity.reshape(-1).long())        
        acc = categorical_accuracy(predictions, batch.polarity.reshape(-1).long())     
        loss.backward()     
        optimizer.step()      
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion):
    
    epoch_loss = 0
    epoch_acc = 0
    
    model.eval() 
    
    with torch.no_grad(): 
    
        for batch in iterator:
            predictions = model(batch)     
            loss = criterion(predictions, batch.polarity.reshape(-1).long())
            acc = categorical_accuracy(predictions, batch.polarity.reshape(-1).long())
            epoch_loss += loss.item()
            epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)
optimizer = torch.optim.Adam(model.parameters())

Entrenem el model.

In [26]:
import time
N_EPOCHS = 5

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    
    start_time = time.time()
    
    train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, valid_iterator, criterion)
    
    end_time = time.time()
        
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
        
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'absa2.pt')
    
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

Epoch: 01 | Epoch Time: 0m 28s
	Train Loss: 0.614 | Train Acc: 71.81%
	 Val. Loss: 0.630 |  Val. Acc: 69.74%
Epoch: 02 | Epoch Time: 0m 26s
	Train Loss: 0.600 | Train Acc: 72.09%
	 Val. Loss: 0.626 |  Val. Acc: 69.74%
Epoch: 03 | Epoch Time: 0m 26s
	Train Loss: 0.599 | Train Acc: 73.08%
	 Val. Loss: 0.641 |  Val. Acc: 69.74%
Epoch: 04 | Epoch Time: 0m 27s
	Train Loss: 0.600 | Train Acc: 73.18%
	 Val. Loss: 0.616 |  Val. Acc: 69.74%
Epoch: 05 | Epoch Time: 0m 28s
	Train Loss: 0.594 | Train Acc: 73.25%
	 Val. Loss: 0.649 |  Val. Acc: 69.74%
